In [10]:
# Install dependencies
!pip install yfinance pandas numpy plotly --quiet

# Imports
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from datetime import datetime

# Factor ETFs and Labels
factor_etfs = {
    "VLUE": "Value",
    "MTUM": "Momentum",
    "SPLV": "Low Volatility",
    "SPY": "Benchmark"
}

# Top Stocks by Factor
factor_top_stocks = {
    "Value": ["JPM", "BRK-B", "META"],
    "Momentum": ["NVDA", "JPM", "AMZN"],
    "Low Volatility": ["MSFT", "BRK-B", "JPM"]
}

# Parameters
include_costs = True
slippage_estimate = 0.0005  # 0.05% slippage
bid_ask_spread = 0.0003     # 0.03% bid-ask
cost_multiplier = 1         # multiplier for turnover estimation

# Date Range
start_date = "2015-01-01"
today = datetime.today()
end_date = today.strftime('%Y-%m-%d')

# Download Data
print("📥 Downloading ETF data...")
raw_etf_data = yf.download(list(factor_etfs.keys()), start=start_date, end=end_date, auto_adjust=True)
etf_data = raw_etf_data['Close'] if 'Close' in raw_etf_data.columns else raw_etf_data

print("📥 Downloading stock data...")
all_stocks = list({s for stocks in factor_top_stocks.values() for s in stocks})
raw_stock_data = yf.download(all_stocks, start=start_date, end=end_date, auto_adjust=True)
stock_data = raw_stock_data['Close'] if 'Close' in raw_stock_data.columns else raw_stock_data

# Resample to Month-End and Forward Fill Current Month if Missing
etf_monthly = etf_data.resample("ME").last().ffill()
stock_monthly = stock_data.resample("ME").last().ffill()
today_pd = pd.to_datetime(today.strftime('%Y-%m-%d'))
if today_pd > etf_monthly.index[-1]:
    etf_monthly.loc[today_pd] = etf_data.iloc[-1]
    stock_monthly.loc[today_pd] = stock_data.iloc[-1]

etf_returns = etf_monthly.pct_change()

# Strategy Logic
strategy_returns = []
best_factors_history = []
last_holdings = set()

for i in range(1, len(etf_returns)):
    prev_month = etf_returns.iloc[i - 1].drop("SPY", errors='ignore')
    if prev_month.isnull().all():
        continue

    best_factor_etf = prev_month.idxmax()
    best_factor_name = factor_etfs.get(best_factor_etf)
    if best_factor_name is None:
        continue

    best_factors_history.append(best_factor_name)
    stocks_to_buy = factor_top_stocks.get(best_factor_name, [])

    try:
        returns = (stock_monthly[stocks_to_buy].iloc[i] / stock_monthly[stocks_to_buy].iloc[i - 1]) - 1
        avg_return = returns.mean()

        # Estimate turnover as percent of different holdings
        turnover = len(set(stocks_to_buy).symmetric_difference(last_holdings)) / len(stocks_to_buy)
        last_holdings = set(stocks_to_buy)

        if include_costs:
            cost = turnover * cost_multiplier * (slippage_estimate + bid_ask_spread)
            avg_return -= cost

        strategy_returns.append(avg_return)
    except:
        continue

# Performance Series
strategy_returns = pd.Series(strategy_returns, index=etf_returns.index[1:len(strategy_returns)+1])
benchmark_returns = etf_returns["SPY"].loc[strategy_returns.index]

cumulative_strategy = (1 + strategy_returns).cumprod()
cumulative_benchmark = (1 + benchmark_returns).cumprod()

# Performance Metrics
def sharpe_ratio(returns, risk_free=0.01):
    excess = returns - (risk_free / 12)
    return np.mean(excess) / np.std(excess) * np.sqrt(12)

def max_drawdown(series):
    peak = series.cummax()
    drawdown = (series / peak) - 1
    return drawdown.min()

strategy_total_return = round((cumulative_strategy.iloc[-1] - 1) * 100, 2)
benchmark_total_return = round((cumulative_benchmark.iloc[-1] - 1) * 100, 2)
cagr = (cumulative_strategy.iloc[-1]) ** (1 / (len(strategy_returns)/12)) - 1
volatility = strategy_returns.std() * np.sqrt(12)
sharpe = sharpe_ratio(strategy_returns)
drawdown = max_drawdown(cumulative_strategy) * 100
calmar = cagr / abs(max_drawdown(cumulative_strategy))

latest_month = strategy_returns.index[-1].strftime('%B %Y')
latest_factor = best_factors_history[-1] if best_factors_history else "N/A"
latest_picks = ", ".join(factor_top_stocks.get(latest_factor, []))

# Drawdowns for plotting
rolling_peak = cumulative_strategy.cummax()
drawdowns = (cumulative_strategy / rolling_peak - 1) * 100

# Strategy Summary (HTML-Formatted)
summary_text = (
    f"Total Return (10Y): {strategy_total_return:.2f}% (Strategy) vs {benchmark_total_return:.2f}% (SPY Benchmark)<br>"
    f"Annualized Sharpe Ratio: {sharpe:.2f}<br>"
    f"Calmar Ratio: {calmar:.2f}<br>"
    f"Maximum Drawdown: {drawdown:.2f}%<br>"
    f"CAGR: {cagr*100:.2f}%<br>"
    f"Annualized Volatility: {volatility*100:.2f}%<br><br>"

    f"🕵️ Stock Picks for {latest_month}:<br>"
    f"Best Performing Factor: {latest_factor}<br>"
    f"Buy: {latest_picks}<br><br>"

    f"Momentum Top 3: {', '.join(factor_top_stocks['Momentum'])}<br>"
    f"Volatility Top 3: {', '.join(factor_top_stocks['Low Volatility'])}<br>"
    f"Value Top 3: {', '.join(factor_top_stocks['Value'])}<br><br>"

    "📜 Strategy Logic: Monthly factor rotation based on trailing 1-month ETF performance. "
    "The top factor ETF guides stock selection from preselected proxies.<br><br>"
    "📚 Theoretical Justification: Empirical studies (e.g., Asness et al. 2013) validate factor persistence. "
    "Our model aligns with this by adaptively capturing the dominant signal. Returns are enhanced via concentrated equity exposure rather than broad ETF allocation.<br><br>"
    "⚠️ Assumptions: No transaction costs, taxes, or liquidity frictions. Stocks selected manually per factor. Results reflect backtest only.<br><br>"
    "🤔 Conclusion: This project demonstrates how quantitative finance leverages simple momentum and cross-sectional alpha concepts to achieve meaningful market outperformance."
)

# Build Interface
fig = make_subplots(
    rows=2, cols=1,
    row_heights=[0.7, 0.3],
    specs=[[{}], [{"type": "table"}]],
    vertical_spacing=0.05
)

# Add Strategy Line
fig.add_trace(go.Scatter(
    x=cumulative_strategy.index,
    y=(cumulative_strategy * 100).round(2),
    name="Strategy",
    line=dict(color="#1f77b4", width=3),
    hovertemplate='%{x|%m/%d/%Y}<br>%{y:.2f}%'
), row=1, col=1)

# Add SPY Benchmark Line
fig.add_trace(go.Scatter(
    x=cumulative_benchmark.index,
    y=(cumulative_benchmark * 100).round(2),
    name="SPY Benchmark",
    line=dict(color="#2ca02c", width=3),
    hovertemplate='%{x|%m/%d/%Y}<br>%{y:.2f}%'
), row=1, col=1)

# Add Drawdown Line
fig.add_trace(go.Scatter(
    x=drawdowns.index,
    y=drawdowns.round(2),
    name="Strategy Drawdown",
    line=dict(color="#d62728", width=2),
    yaxis="y2",
    hovertemplate='%{x|%m/%d/%Y}<br>%{y:.2f}%'
), row=1, col=1)

# Summary Text as Table
fig.add_trace(go.Table(
    columnwidth=[3],
    header=dict(values=["Strategy Summary"], fill_color='black', font=dict(size=16, color='white'), align='left'),
    cells=dict(values=[[summary_text]], fill_color='black', align='left', font=dict(size=12, color='white'))
), row=2, col=1)

# Layout
fig.update_layout(
    title="📈 Adaptive Factor Rotation Strategy vs SPY (Backtest)",
    xaxis_title="Date",
    yaxis=dict(title="Cumulative Return (%)", side='left', overlaying="y2"),
    yaxis2=dict(title="Strategy Drawdown (%)", side='right', showgrid=False),
    height=1000,
    template="plotly_dark",
    margin=dict(l=40, r=40, t=80, b=40),
    showlegend=True,
    xaxis=dict(
        rangeselector=dict(
            buttons=list([
                dict(count=1, label="1M", step="month", stepmode="backward"),
                dict(count=3, label="3M", step="month", stepmode="backward"),
                dict(count=6, label="6M", step="month", stepmode="backward"),
                dict(count=1, label="YTD", step="year", stepmode="todate"),
                dict(count=1, label="1Y", step="year", stepmode="backward"),
                dict(step="all", label="10Y")
            ]),
            bgcolor="#111",
            activecolor="#333",
            font=dict(color="white")
        ),
        rangeslider=dict(visible=False)
    )
)

fig.show()


📥 Downloading ETF data...


[*********************100%***********************]  4 of 4 completed


📥 Downloading stock data...


[*********************100%***********************]  6 of 6 completed
